# B · Who they invest in (recipients)
**Question:** *Who is the SBIR/STTR money actually going to — and how concentrated is it?*

Reads `data/tx_sbir_clean.parquet` from `01_ingest_clean`. Award data only — no external source.
> **Source:** SBIR/STTR Award Data (U.S. SBA), Texas, 2016–2025. See `../SOURCES.md`.

In [ ]:
# --- Setup & house style (Colab-friendly) ---
# !pip install -q pandas numpy pyarrow matplotlib
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

try:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/TX_SBIR_STTR_Analysis'
except Exception:
    BASE = '..'
DATA_DIR = os.environ.get('DATA_DIR', f'{BASE}/data')
OUT = os.environ.get('OUT_DIR', f'{BASE}/outputs'); os.makedirs(OUT, exist_ok=True)
df = pd.read_parquet(os.path.join(DATA_DIR, 'tx_sbir_clean.parquet'))
YR='award_year'
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':120,'font.size':11,
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,
    'grid.color':'#e6e6e6','grid.linewidth':0.8,'axes.axisbelow':True,'axes.edgecolor':'#888'})
BLUE, ORANGE, GRAY = '#0072B2', '#E69F00', '#9aa0a6'
usd = FuncFormatter(lambda v,_: f'${v/1e6:,.0f}M' if v>=1e6 else f'${v:,.0f}')
def save(fig,name): fig.savefig(os.path.join(OUT,name), bbox_inches='tight', facecolor='white'); return fig
# One display name per firm_key (most common spelling)
firm = pd.DataFrame({
    'name': df.groupby('firm_key')['company_name'].agg(lambda s: s.mode().iat[0]),
    'awards': df.groupby('firm_key').size(),
    'usd': df.groupby('firm_key')['award_amount_num'].sum(),
})
print(f'{len(firm):,} unique firms across {len(df):,} awards')

## The most prolific firms (by award count)
**Methodology:** count of award rows per `firm_key` (dedup on UEI, else normalized name).
**Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- Top 15 firms by number of awards ---
t = firm.sort_values('awards', ascending=True).tail(15)
fig, ax = plt.subplots(figsize=(9,5.5))
bars = ax.barh(t['name'], t['awards'], color=BLUE)
for b,v in zip(bars,t['awards']): ax.text(v+3, b.get_y()+b.get_height()/2, f'{v}', va='center', fontsize=9, color='#333')
ax.set_xlabel('Number of SBIR/STTR awards, 2016–2025'); ax.set_xlim(0, t['awards'].max()*1.12)
ax.set_title('Most prolific Texas SBIR/STTR firms (by award count)', fontweight='bold', loc='left')
save(fig,'B1_top_firms_count.png'); plt.show()

### What this means
A **handful of firms dominate award *count*** — led by **Lynntech (335 awards)**, then Nanohmics
(182) and Texas Research Institute (141). These are classic **serial SBIR winners**: they run many
small projects at once. Whether that converts to commercialization (vs. the “SBIR-mill” pattern) is
tested in Module E.

## Top firms by dollars
**Methodology:** sum of `award_amount_num` per firm. **Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- Top 15 firms by dollars ---
t = firm.sort_values('usd', ascending=True).tail(15)
fig, ax = plt.subplots(figsize=(9,5.5))
bars = ax.barh(t['name'], t['usd'], color=BLUE)
for b,v in zip(bars,t['usd']): ax.text(v+t['usd'].max()*0.01, b.get_y()+b.get_height()/2, f'${v/1e6:.0f}M', va='center', fontsize=9, color='#333')
ax.xaxis.set_major_formatter(usd); ax.set_xlim(0, t['usd'].max()*1.15)
ax.set_title('Top Texas SBIR/STTR firms by award dollars', fontweight='bold', loc='left')
save(fig,'B2_top_firms_usd.png'); plt.show()

### What this means
The dollar ranking is **less lopsided than the count ranking**. Lynntech leads both, but firms like
**Wilder Systems and HTX Labs** appear here with *few* awards yet large dollars — they win bigger
**Phase II** awards rather than many small Phase I's. Two different strategies show up in the same data.

## How many awards does a firm win?
**Methodology:** distribution of awards-per-firm, bucketed. **Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- How many awards does a firm win? (repeat-winner distribution) ---
buckets = pd.cut(firm['awards'], [0,1,2,4,9,49,10000],
                 labels=['1','2','3–4','5–9','10–49','50+'])
bc = buckets.value_counts().reindex(['1','2','3–4','5–9','10–49','50+'])
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(bc.index.astype(str), bc.values, color=BLUE, width=0.7)
ax.bar_label(bars, padding=2, fontsize=9, color='#333')
ax.set_xlabel('Awards won by a firm (2016–2025)'); ax.set_ylabel('Number of firms')
ax.set_title('Most firms win 1–2 awards; a few win dozens', fontweight='bold', loc='left')
save(fig,'B3_awards_per_firm.png'); plt.show()

### What this means
**Most Texas firms are occasional winners** (the median firm has 2 awards), but a long tail of
**~56 firms won 10+** and a few won dozens. The program is broad at the base and highly concentrated
at the top — quantified next.

## New vs. returning awardees
**Methodology:** a firm is **new** the first year it appears *within the 2016–2025 window*.
**Limitation:** firms with pre-2016 awards are counted “new to window,” not new to SBIR — the full
file goes back to 1983 and a lifetime-first-award version can be built if wanted. **Source:** SBA, TX.

In [ ]:
# --- New vs returning awardees each year ---
# 'New' = firm's FIRST appearance within the 2016-2025 window (see limitation below).
first_year = df.groupby('firm_key')[YR].min()
years = list(range(int(df[YR].min()), int(df[YR].max())+1))
rows=[]
for y in years:
    active = df[df[YR]==y]['firm_key'].unique()
    new = sum(first_year[f]==y for f in active)
    rows.append((y, new, len(active)-new))
nf = pd.DataFrame(rows, columns=['year','new','returning']).set_index('year')
fig, ax = plt.subplots(figsize=(11,4.2))
ax.bar(nf.index, nf['returning'], color=BLUE, label='Returning firm')
ax.bar(nf.index, nf['new'], bottom=nf['returning'], color=ORANGE, label='New to window')
ax.set_xticks(years); ax.set_ylabel('Distinct firms funded')
ax.set_title('New vs. returning awardees each year', fontweight='bold', loc='left')
ax.legend(frameon=False, loc='upper left')
save(fig,'B4_new_vs_returning.png'); plt.show()

### What this means
Every year brings a **fresh cohort of first-time (in-window) firms** on top of a growing base of
**returning** winners — the program is both renewing its pipeline and retaining prior awardees, not
just recycling the same names.

## How concentrated is the money?
**Methodology:** Lorenz curve of award dollars across firms; **Gini** = area between the curve and
the equality line (0 = every firm equal, 1 = one firm has everything). **Source:** SBA, TX.

In [ ]:
# --- Concentration: how much of the money goes to how few firms (Lorenz curve) ---
v = np.sort(firm['usd'].values)
cum = np.cumsum(v)/v.sum()
n = len(v); x = np.arange(1, n+1)/n
# closed-form Gini (no np.trapz; robust across numpy versions)
gini = (2*np.sum(np.arange(1, n+1)*v) - (n+1)*np.sum(v)) / (n*np.sum(v))
fig, ax = plt.subplots(figsize=(6.2,6))
ax.plot([0,1],[0,1], ls='--', color=GRAY, lw=1.5, label='Perfect equality')
ax.plot(x, cum, color=BLUE, lw=2.4, label='Actual')
ax.fill_between(x, cum, x, color=BLUE, alpha=0.08)
ax.set_xlabel('Cumulative share of firms (poorest → richest)')
ax.set_ylabel('Cumulative share of award dollars')
ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f'Award dollars are concentrated (Gini ≈ {gini:.2f})', fontweight='bold', loc='left')
ax.legend(frameon=False, loc='upper left')
save(fig,'B5_lorenz.png'); plt.show()
top10 = 100*firm['usd'].sort_values(ascending=False).head(10).sum()/firm['usd'].sum()
print(f'Top 10 firms hold {top10:.0f}% of all TX SBIR/STTR dollars.')

### What this means
Award dollars are **highly concentrated** — the **top 10 firms hold ~25%** of all Texas SBIR/STTR
dollars, and the Gini is high. This is expected for R&D funding (winners compound), but it frames a
policy question the SBDC cares about: is the goal to **deepen** a few champions or **widen** the base?

## Who owns the winning firms?
**Methodology:** SBIR.gov self-reported ownership flags (Y/N/U). We report **Y as a share of all**
awards/dollars (unknowns left in the base → conservative). **Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- Demographic ownership: share of awards & dollars ---
# Flags are Y / N / U(unknown). We report Y as a share of ALL awards; 'U' stays in the base
# (so these are conservative lower bounds). Denominator noted on the chart.
flags = [('women_owned','Woman-owned'), ('socially_economically_disadvantaged','Disadvantaged'), ('hubzone_owned','HUBZone')]
labels, aw_share, usd_share = [], [], []
for colname,lab in flags:
    y = df[colname].astype(str).str.upper().eq('Y')
    labels.append(lab)
    aw_share.append(100*y.mean())
    usd_share.append(100*df.loc[y,'award_amount_num'].sum()/df['award_amount_num'].sum())
x=np.arange(len(labels)); w=0.38
fig, ax = plt.subplots(figsize=(8,4))
b1=ax.bar(x-w/2, aw_share, w, color=GRAY, label='% of awards')
b2=ax.bar(x+w/2, usd_share, w, color=BLUE, label='% of dollars')
ax.bar_label(b1, fmt='%.1f%%', padding=2, fontsize=9); ax.bar_label(b2, fmt='%.1f%%', padding=2, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('Percent of TX total')
ax.set_ylim(0, max(max(aw_share),max(usd_share))*1.3)
ax.set_title('Awards to under-represented-owned firms', fontweight='bold', loc='left')
ax.legend(frameon=False)
save(fig,'B6_demographics.png'); plt.show()

### What this means
**Woman-owned (~8–9%), socially/economically disadvantaged (~9%), and HUBZone (~2–3%)** firms win a
modest share of Texas awards — and their **dollar** share is similar to or below their award share.
This is a concrete equity baseline the SBDC can target with Phase-0 outreach.

---
*Next: **Module D · Geography** (unique firms by county — needs the county-crosswalk source), then
**C · Growth**, **E · Commercialization funnel**, **F · Survival**, **G · Impact benchmarking**.*